## setup

In [1]:
import os, cv2
import numpy as np
import pandas as pd

In [2]:
F_DATA_IMAGES = "data/all/some-removed/augmented-traditional/images"
F_DATA_MASKS = "data/all/some-removed/augmented-traditional/masks"

CSV_PATH = "data/all/some-removed/augmented-traditional/data.csv"
open(CSV_PATH, "w").close()

In [3]:
class DataExtractor:

    def __init__(self, csv_path, color_left_lung, color_right_lung, color_heart):
        self.csv_path = csv_path
        self.color_left_lung, self.color_right_lung, self.color_heart = color_left_lung, color_right_lung, color_heart

    def get_df(self, n=None):
        df = pd.read_csv(self.csv_path)
        return df.head(n) if n is not None else df

    def extract_names(self, folder):
        names = sorted([
            os.path.splitext(f)[0]
            for f in os.listdir(folder)
            if os.path.isfile(os.path.join(folder, f))
        ])
        df = pd.DataFrame({"name": names})
        df.to_csv(self.csv_path, index=False)

    def extract_all_widths_and_ctr(self, masks_folder):
        df = self.get_df()

        for idx, row in df.iterrows():
            base_name = row['name']
            
            img_path = None
            for filename in os.listdir(masks_folder):
                if os.path.splitext(filename)[0] == base_name:
                    img_path = os.path.join(masks_folder, filename)
                    break
            
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            cardiac_width = None
            heart_mask = np.all(img == self.color_heart, axis=2)
            if np.any(heart_mask):
                heart_coords = np.where(heart_mask)
                heart_x_coords = heart_coords[1]
                cardiac_width = np.max(heart_x_coords) - np.min(heart_x_coords) + 1
            
            thoracic_width = 0
            left_lung_mask = np.all(img == self.color_left_lung, axis=2)
            right_lung_mask = np.all(img == self.color_right_lung, axis=2)
            left_lung_coords = np.where(left_lung_mask)
            right_lung_coords = np.where(right_lung_mask)
            
            if len(left_lung_coords[0]) > 0 and len(right_lung_coords[0]) > 0:
                all_y_coords = np.unique(np.concatenate([left_lung_coords[0], right_lung_coords[0]]))
                
                max_distance = 0
                for y in all_y_coords:
                    left_lung_x_at_y = left_lung_coords[1][left_lung_coords[0] == y]
                    right_lung_x_at_y = right_lung_coords[1][right_lung_coords[0] == y]
                    
                    if len(left_lung_x_at_y) > 0 and len(right_lung_x_at_y) > 0:
                        leftmost_left = np.min(left_lung_x_at_y)
                        rightmost_right = np.max(right_lung_x_at_y)
                        distance = rightmost_right - leftmost_left + 1
                        max_distance = max(max_distance, distance)
                
                thoracic_width = max_distance
            
            df.loc[idx, 'thoracic_width'] = thoracic_width if thoracic_width > 0 else None
            df.loc[idx, 'cardiac_width'] = cardiac_width
            
            if cardiac_width is not None and thoracic_width is not None and thoracic_width > 0:
                df.loc[idx, 'ctr'] = f"{cardiac_width / thoracic_width:.3f}"
            else:
                df.loc[idx, 'ctr'] = None
        
        df.to_csv(self.csv_path, index=False)

    def extract_cardiomegaly(self):
        df = self.get_df()
        df['cardiomegaly'] = (df['ctr'] > 0.5).astype(int)
        df.to_csv(self.csv_path, index=False)

    def extract_genders(self):
        df = self.get_df()

        def assign_gender(name):
            if "-M-" in name:
                return "M"
            elif "-F-" in name:
                return "F"
            else:
                return ""

        df['gender'] = df['name'].apply(assign_gender)
        df.to_csv(self.csv_path, index=False)
        
    def _crop_padding(self, img):
        if img is None:
            return None
        
        h, w = img.shape
        mean_rows = np.mean(img, axis=1)
        mean_cols = np.mean(img, axis=0)
        low_thresh, high_thresh = 15, 240

        rows_to_keep = (mean_rows >= low_thresh) & (mean_rows <= high_thresh)
        if np.any(rows_to_keep):
            first_row = np.argmax(rows_to_keep)
            last_row = len(rows_to_keep) - np.argmax(rows_to_keep[::-1]) - 1
        else:
            first_row, last_row = 0, h - 1
        
        cols_to_keep = (mean_cols >= low_thresh) & (mean_cols <= high_thresh)
        if np.any(cols_to_keep):
            first_col = np.argmax(cols_to_keep)
            last_col = len(cols_to_keep) - np.argmax(cols_to_keep[::-1]) - 1
        else:
            first_col, last_col = 0, w - 1
        
        if last_row >= first_row and last_col >= first_col:
            return img[first_row:last_row+1, first_col:last_col+1]
        else:
            return img

    def extract_brightness(self, images_folder):
        df = self.get_df()
        brightness_list = []

        for idx, row in df.iterrows():
            image_name = row['name']
            image_path = os.path.join(images_folder, f"{image_name}.png")
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                brightness = np.nan
            else:
                cropped_img = self._crop_padding(img)
                if cropped_img is not None:
                    brightness = np.mean(cropped_img)
                else:
                    brightness = np.nan
            
            brightness_list.append(f"{brightness:.1f}")

        df['brightness'] = brightness_list
        df.to_csv(self.csv_path, index=False)


extractor = DataExtractor(CSV_PATH, (85,85,85), (170,170,170), (255,255,255))

In [4]:
extractor.extract_names(F_DATA_IMAGES)
extractor.get_df()

,name
0,jsrt-JPCLN001
1,jsrt-JPCLN001_aug_grid_distortion_0
2,jsrt-JPCLN002
3,jsrt-JPCLN002_aug_gaussian_blur_0
4,jsrt-JPCLN003
...,...
2071,shenzhen-M-1561_aug_grid_distortion_0
2072,shenzhen-M-1564
2073,shenzhen-M-1564_aug_grid_distortion_0
2074,shenzhen-M-1565


## CTR values

In [5]:
extractor.extract_all_widths_and_ctr(F_DATA_MASKS)
extractor.extract_cardiomegaly()
extractor.get_df(5)

,name,thoracic_width,cardiac_width,ctr,cardiomegaly
0,jsrt-JPCLN001,389.0,184.0,0.473,0
1,jsrt-JPCLN001_aug_grid_distortion_0,384.0,182.0,0.474,0
2,jsrt-JPCLN002,362.0,222.0,0.613,1
3,jsrt-JPCLN002_aug_gaussian_blur_0,362.0,222.0,0.613,1
4,jsrt-JPCLN003,371.0,184.0,0.496,0


## genders

In [6]:
extractor.extract_genders()

## brightness

In [7]:
extractor.extract_brightness(F_DATA_IMAGES)
extractor.get_df(5)

,name,thoracic_width,cardiac_width,ctr,cardiomegaly,gender,brightness
0,jsrt-JPCLN001,389.0,184.0,0.473,0,NaN,196.9
1,jsrt-JPCLN001_aug_grid_distortion_0,384.0,182.0,0.474,0,NaN,198.4
2,jsrt-JPCLN002,362.0,222.0,0.613,1,NaN,160.2
3,jsrt-JPCLN002_aug_gaussian_blur_0,362.0,222.0,0.613,1,NaN,160.4
4,jsrt-JPCLN003,371.0,184.0,0.496,0,NaN,206.9


## new